xenium hbc

导入相关包

In [1]:
from pathlib import Path
import warnings
import os
import scanpy as sc
import scib
import numpy as np
import pandas as pd
import sys
import scgpt as scg
import matplotlib.pyplot as plt
import anndata

import time
import torch
from memory_profiler import memory_usage
from datetime import datetime
from functools import wraps

plt.style.context('default')
warnings.simplefilter("ignore", ResourceWarning)

model_dir = Path("/home/cavin/jt/benchmark/models/SCGPT")


def measure_resources(cuda_id: int = 0,task_des:str="task",save_dir:str="/home/cavin/jt/benchmark/experiments/results/run_metric/spatial_cluster_no_annotations",save_file_name:str="record.csv"):
    """
    测量函数运行时的CPU内存、指定GPU的显存消耗(峰值)和运行时间。
    
    参数:
    cuda_id (int): 需要监控的 GPU 编号，例如传入 0 代表监控 cuda:0
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            
            
            device = None
            if torch.cuda.is_available():
                try:
                    if not torch.cuda.is_initialized():
                        torch.cuda.init()
                    device = torch.device(f"cuda:{cuda_id}")
                    torch.cuda.reset_peak_memory_stats(device)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 初始化监控失败: {e}")

            start_time = time.time()
            
            mem_usage_mb, result = memory_usage(
                (func, args, kwargs), 
                max_usage=True, 
                retval=True
            )
            
            end_time = time.time()
            execution_time = end_time - start_time
            
           
            mem_usage_gb = mem_usage_mb / 1024.0
            
            allocated_gb = 0.0
            cached_gb = 0.0

            
            if device is not None:
                try:
                    allocated_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
                    cached_gb = torch.cuda.max_memory_reserved(device) / (1024 ** 3)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 显存统计失败: {e}")

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # ===========================
            # 美观的表格式输出
            # ===========================
            report = f"""
============================================================
RESOURCE USAGE REPORT: {func.__name__}
============================================================
Timestamp                   : {timestamp}
Target GPU ID               : {cuda_id}
Execution Time (Minutes)    : {execution_time/60:.4f}
Execution Time (Seconds)    : {execution_time:.2f}
CPU Peak Memory Usage (GB)  : {mem_usage_gb:.4f}
GPU Peak Allocated Mem (GB) : {allocated_gb:.4f}
GPU Peak Cached Mem (GB)    : {cached_gb:.4f}
CUDA Available              : {torch.cuda.is_available()}
============================================================
"""
            print(report)
            full_path = os.path.join(save_dir, save_file_name)
            os.makedirs(save_dir, exist_ok=True)
            new_record = {"Timestamp":timestamp,"Task":task_des,
                          "Target GPU ID ":cuda_id,
                          "Execution Time (Minutes)":execution_time / 60,
                          "Execution Time (Seconds)":execution_time,
                          "CPU Peak Memory Usage (GB)":mem_usage_gb,
                          "GPU Peak Allocated Mem (GB)":allocated_gb,
                          "GPU Peak Cached Mem (GB)":cached_gb}
            new_data_df = pd.DataFrame([new_record])
            if not os.path.isfile(full_path):
                # 如果文件不存在：直接正常写入（默认 mode='w'），自带表头
                new_data_df.to_csv(full_path, index=False)
                print(f"文件不存在，已创建新文件并写入表头：{full_path}")
            else:
                # 如果文件已存在：使用追加模式 (mode='a')，并且绝对不写表头 (header=False)
                new_data_df.to_csv(full_path, mode='a', header=False, index=False)
                print(f"文件已存在，数据已成功追加至末尾。")
            return result
        return wrapper
    return decorator

读取数据和数据预处理

In [ ]:
shard_inx = "0"
simple_path = f'/home/cavin/jt/benchmark/data/crc/VisiumHD_P2_shard_{shard_inx}.h5ad'
adata = sc.read_h5ad(simple_path)
gene_col = "gene_name"
# cell_type_key = "cell_type"
# celltype_id_labels = adata.obs[cell_type_key].astype("category").cat.codes.values
# adata = adata[celltype_id_labels >= 0]
org_adata = adata.copy()
adata

AnnData object with n_obs × n_vars = 166824 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial'

In [3]:
adata.var_names

Index(['ENSG00000187608', 'ENSG00000186891', 'ENSG00000186827',
       'ENSG00000162576', 'ENSG00000205116', 'ENSG00000179403',
       'ENSG00000142609', 'ENSG00000187730', 'ENSG00000116254',
       'ENSG00000049249',
       ...
       'ENSG00000102245', 'ENSG00000165370', 'ENSG00000101981',
       'ENSG00000165509', 'ENSG00000155495', 'ENSG00000268089',
       'ENSG00000182492', 'ENSG00000130821', 'ENSG00000198910',
       'ENSG00000286265'],
      dtype='object', length=2000)

提取嵌入

In [ ]:
@measure_resources(task_des=f"SCGpt run visiumHD human CRC P2 shard {shard_inx}",save_file_name=f"shard_{shard_inx}_record.csv")
def get_emb():
    embed_adata = scg.tasks.embed_data(
        adata,
        model_dir,
        gene_col=gene_col,
        batch_size=16,
    )
    return embed_adata.obsm["X_scGPT"]

In [5]:
emb = get_emb()

scGPT - INFO - match 1971/2000 genes in vocabulary of size 60697.


Embedding cells: 100%|██████████| 10427/10427 [01:08<00:00, 152.81it/s]



RESOURCE USAGE REPORT: get_emb
Timestamp                   : 2026-03-05 13:58:36
Target GPU ID               : 0
Execution Time (Minutes)    : 1.1611
Execution Time (Seconds)    : 69.66
CPU Peak Memory Usage (GB)  : 4.3050
GPU Peak Allocated Mem (GB) : 0.2882
GPU Peak Cached Mem (GB)    : 0.3184
CUDA Available              : True

文件不存在，已创建新文件并写入表头：/home/cavin/jt/benchmark/experiments/results/run_metric/spatial_cluster_no_annotations/shard_2_record.scv


/home/cavin/anaconda3/envs/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


保存嵌入

In [6]:
emb.shape

(166824, 512)

In [ ]:
emb_df = pd.DataFrame(
    emb, 
    index=adata.obs_names,
    columns=[f"scGPT_dim_{i}" for i in range(emb.shape[1])]
)
save_path = f"/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_no_annotations/scgpt_human_CRC_shard_{shard_inx}_emb.parquet"
emb_df.to_parquet(save_path, compression="zstd")

读取嵌入

In [8]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 166824 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial'

In [9]:
adata.obsm["X_scGPT"] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 166824 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial', 'X_scGPT'